# Intro

Download all the quicklook files from AirHARP for PACE-PAX from their website

PACE-PAX  
AirHARP  
Quicklooks   

# Load the python files

In [2]:
import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Build the different functions

In [1]:


def get_rendered_html(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(url)
    time.sleep(3)  # wait for JS to render
    html = driver.page_source
    driver.quit()
    return html

def get_subfolders(base_url):
    html = get_rendered_html(base_url)
    soup = BeautifulSoup(html, "html.parser")
    folders = [p.text.strip() for p in soup.find_all('p', class_='name') if p.text.strip().isdigit()]
    return folders

def get_file_links(folder_url):
    html = get_rendered_html(folder_url)
    soup = BeautifulSoup(html, "html.parser")
    anchors = soup.find_all("a", class_="list-item")
    return [urljoin(folder_url, a["href"].split("/")[-1]) for a in anchors if a["href"].endswith(".png")]

def download_file(url, local_folder):
    os.makedirs(local_folder, exist_ok=True)
    filename = os.path.basename(url)
    filepath = os.path.join(local_folder, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename} ...")
        r = requests.get(url)
        with open(filepath, 'wb') as f:
            f.write(r.content)

def main():
    base_url = "https://aether.esi-nyx-mobile.cloud/pace-pax/Polarimeter/quicklook-l1c/"
    output_dir = "quicklook_downloads"

    folders = get_subfolders(base_url)
    print(f"Found folders: {folders}")

    for folder in folders:
        folder_url = urljoin(base_url, folder + "/")
        print(f"Processing folder: {folder_url}")
        file_links = get_file_links(folder_url)
        for link in file_links:
            full_url = urljoin(folder_url, os.path.basename(link))
            download_file(full_url, os.path.join(output_dir, folder))

Found folders: ['20240829', '20240904', '20240906', '20240908', '20240910', '20240913', '20240915', '20240917', '20240922', '20240923', '20240926', '20240927', '20240929', '20240930']
Processing folder: https://aether.esi-nyx-mobile.cloud/pace-pax/Polarimeter/quicklook-l1c/20240829/
Processing folder: https://aether.esi-nyx-mobile.cloud/pace-pax/Polarimeter/quicklook-l1c/20240904/


KeyboardInterrupt: 

In [ ]:
import os
import time
import requests
from urllib.parse import urljoin
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

BASE_URL = "https://aether.esi-nyx-mobile.cloud/pace-pax/Polarimeter/quicklook-l1c/"
OUTPUT_ROOT = "quicklook_downloads"

def setup_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def get_rendered_html(driver, url):
    driver.get(url)
    time.sleep(3)
    return BeautifulSoup(driver.page_source, "html.parser")

def get_folder_list(driver, base_url):
    soup = get_rendered_html(driver, base_url)
    folders = [p.text.strip() for p in soup.find_all('p', class_='name') if p.text.strip().isdigit()]
    return folders

def get_signed_file_links(driver, folder_url):
    soup = get_rendered_html(driver, folder_url)
    anchors = soup.find_all("a", class_="list-item")
    return [urljoin(folder_url, a.get("href")) for a in anchors if a.get("href") and ".png" in a.get("href")]

def download_file(url, local_folder):
    os.makedirs(local_folder, exist_ok=True)
    filename = url.split("/")[-1].split("?")[0]
    local_path = os.path.join(local_folder, filename)
    if os.path.exists(local_path):
        print(f"Already exists: {filename}")
        return
    print(f"Downloading {filename} ...")
    r = requests.get(url)
    with open(local_path, 'wb') as f:
        f.write(r.content)

def main():
    driver = setup_driver()

    try:
        folders = get_folder_list(driver, BASE_URL)
        print(f"Found folders: {folders}")

        for folder in folders:
            folder_url = urljoin(BASE_URL, folder + "/")
            print(f"\nProcessing: {folder_url}")
            links = get_signed_file_links(driver, folder_url)
            if not links:
                print(f"  No files found.")
                continue
            output_path = os.path.join(OUTPUT_ROOT, folder)
            for link in links:
                download_file(link, output_path)

    finally:
        driver.quit()

if __name__ == "__main__":
    main()


In [ ]:
if __name__ == "__main__":
    main()